# Архитектура как набор блоков

Формы тензоров, residual, normalization и проверка градиентов.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

Блок задаёт преобразование тензора. Residual складывает вход и результат блока: формы должны совпадать. LayerNorm нормализует признаки; MLP меняет представление. Внимание добавляет обмен информацией между позициями и требует правильной causal mask.

В Model Builder можно собрать граф, открыть справку по блокам, проверить формы и передать поддерживаемую архитектуру в обучение LLM/VLM/VLA. Не каждый произвольный граф подходит всем задачам: важны modality, causal свойства и выходная размерность. Ниже проверяем простой residual MLP, не заменяя валидатор конструктора.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
class ResidualMLP(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.mlp = nn.Sequential(nn.Linear(width, width * 2), nn.GELU(), nn.Linear(width * 2, width))
    def forward(self, x):
        return x + self.mlp(self.norm(x))
block = ResidualMLP(16)
x = torch.randn(2, 8, 16, requires_grad=True)
y = block(x); y.square().mean().backward()
print('input:', tuple(x.shape), 'output:', tuple(y.shape))
print('parameters:', sum(p.numel() for p in block.parameters()))
print('input gradient norm:', x.grad.norm().item())
assert x.shape == y.shape and torch.isfinite(x.grad).all()

## Попробуйте сами

Поменяйте последнюю Linear на выход ширины 8 и прочитайте ошибку residual. Затем восстановите форму и сравните gradient norm с LayerNorm и без неё.

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [Residual Learning](https://arxiv.org/abs/1512.03385)
- [Layer Normalization](https://arxiv.org/abs/1607.06450)